# 01_entity — Entity: a domain object independent of storage

In an ordinary project the business object is whatever the database returned — an ORM row named after columns, or a foreign API's JSON. AOA declares the business object once, in domain terms, as an **Entity**: a `Resource` hydrates it from flat storage data with `build(...)`, and one Entity serves many load levels via `partial(...)`. Reading a declared-but-unloaded field raises `FieldNotLoadedError` — never a silent `None`.

**What's new**

| Concept | Description |
|---------|-------------|
| `@entity` + `BaseEntity` | a domain object; class name must end with `Entity` |
| `build(data, Entity)` | direct hydration (keys = fields) |
| `build(row, Entity, mapper)` | map storage columns → entity fields (the Resource's only job) |
| `Entity.partial(...)` | materialise only some fields |
| `FieldNotLoadedError` | reading a declared-but-unloaded field — not `None` |

In [ ]:
!pip install aoa-action-machine

In [ ]:
from pydantic import Field

from aoa.action_machine.domain import BaseEntity, FieldNotLoadedError, build
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entity import entity

## The entity — declared in domain terms

`OrderEntity` knows an order has `id`, `total`, `status`. It knows nothing about a table `orders`, a column `total_amount`, or a `JOIN`. (`status` is a plain field here; `Lifecycle` is a later chapter.)

In [ ]:
class ShopDomain(BaseDomain):
    name = "shop"
    description = "Shop domain"


@entity(description="Customer order", domain=ShopDomain)
class OrderEntity(BaseEntity):
    id: str = Field(description="Order identifier")
    total: float = Field(ge=0, description="Order total")
    currency: str = Field(default="RUB", description="Currency code")
    status: str = Field(description="Order status")  # plain field; Lifecycle is a later chapter

## Run — one Entity, three load levels

Full hydration (keys = fields), the same Entity from a differently-shaped row (mapper), and a partial load where reading the unloaded `status` raises `FieldNotLoadedError`.

> This example is synchronous — no `await` needed.

In [ ]:
def main() -> None:
    # 1) Full hydration — keys match fields directly (the simple case).
    order = build({"id": "ord-1", "total": 1500.0, "currency": "RUB", "status": "paid"}, OrderEntity)
    print("1) Full entity (direct build):")
    print(f"   {order.id}  total={order.total} {order.currency}  status={order.status}")

    # 2) Storage-independent: a row with different column names, mapped to the
    #    same Entity. This mapping is the Resource's only job — no business logic.
    row = {"order_id": "ord-2", "total_amount": 99.5, "ccy": "EUR", "state": "shipped"}
    mapped = build(row, OrderEntity, lambda e, r: {
        e.id: r["order_id"],
        e.total: r["total_amount"],
        e.currency: r["ccy"],
        e.status: r["state"],
    })
    print("\n2) Same Entity from a differently-shaped row (mapper):")
    print(f"   {mapped.id}  total={mapped.total} {mapped.currency}  status={mapped.status}")

    # 3) Partial load — only id and total were read from storage.
    print("\n3) Partial load (only id, total):")
    partial = OrderEntity.partial(id="ord-3", total=42.0)
    print(f"   id={partial.id}  total={partial.total}")
    print(f"   is_field_loaded('status') = {partial.is_field_loaded('status')}")
    print(f"   primary key = {partial.get_primary_key()}")
    try:
        _ = partial.status  # declared but not loaded
    except FieldNotLoadedError as exc:
        print(f"   reading status -> FieldNotLoadedError: {exc}")


main()